# Lab 4: Data Wrangling (Real Estate Market)
**Aim:** Data wrangling on California Housing dataset, cleaning columns, handling missing values, merging, filtering, and IQR outlier detection.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

# 1. Load dataset & clean column names
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()
df = df.rename(columns={'medhouseval': 'house_price'}).fillna(df.median(numeric_only=True))
print("Clean column names:", df.columns.tolist())
df.head()

Clean column names: ['medinc', 'houseage', 'averooms', 'avebedrms', 'population', 'aveoccup', 'latitude', 'longitude', 'house_price']


,medinc,houseage,averooms,avebedrms,population,aveoccup,latitude,longitude,house_price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
# 2. Feature Engineering & Merge
df['property_type'] = np.where(df['averooms'] > 5, 'Large', 'Small')
df['income_group'] = pd.cut(df['medinc'], bins=[0, 2.5, 5, np.inf], labels=['Low', 'Medium', 'High'])

neighborhood_info = pd.DataFrame({
    'income_group': ['Low', 'Medium', 'High'],
    'amenity_score': [4, 7, 9]
})
df = df.merge(neighborhood_info, on='income_group', how='left')

# Filter: > 5 rooms and price > 2
filtered_df = df[(df['averooms'] > 5) & (df['house_price'] > 2)]
print("Filtered count:", len(filtered_df))
filtered_df.head()

Filtered count: 5978


,medinc,houseage,averooms,avebedrms,population,aveoccup,latitude,longitude,house_price,property_type,income_group,amenity_score
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,Large,High,9
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,Large,High,9
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,Large,High,9
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,Large,High,9
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,Large,Medium,7


In [3]:
# 3. Encoding & IQR Outlier Removal
df_encoded = pd.get_dummies(df, columns=['property_type'], drop_first=True)

Q1 = df['house_price'].quantile(0.25)
Q3 = df['house_price'].quantile(0.75)
IQR = Q3 - Q1
lower_lim = Q1 - 1.5 * IQR
upper_lim = Q3 + 1.5 * IQR

clean_df = df[(df['house_price'] >= lower_lim) & (df['house_price'] <= upper_lim)].copy()
print("Rows before removing outliers:", len(df))
print("Rows after removing outliers :", len(clean_df))

clean_df.to_csv('Cleaned_RealEstate_Prices.csv', index=False)
print("Cleaned real estate data saved.")

Rows before removing outliers: 20640
Rows after removing outliers : 19569
Cleaned real estate data saved.
